# 03 — Preprocessing & Scaling
### NexaTel Customer Churn Prediction Project

**Goal:** prepare the data correctly for modeling, and do every step in the order that avoids leakage.

**Rule followed throughout this notebook: split first, fit second.** The scaler/encoder is fit on the training set only, then used to transform both train and test — never the other way around.


In [1]:
import sys, os
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

from ml.feature_engineering import NUMERIC_FEATURES, BINARY_FEATURES, NOMINAL_CATEGORICAL_FEATURES, ALL_MODEL_COLUMNS

df = pd.read_csv('../data/processed/features_engineered.csv')
print(df.shape)
df[ALL_MODEL_COLUMNS + ['churn']].head()

(7043, 27)


,tenure,monthly_charges,total_charges,total_services,avg_monthly_spend_ratio,contract_ordinal,senior_citizen,partner,dependents,paperless_billing,...,tenure_group,multiple_lines,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,phone_service,churn
0,1,29.85,29.85,1,29.85,0,False,True,False,True,...,0-12,No phone service,No,Yes,No,No,No,No,No,False
1,34,56.95,1889.50,2,55.57,1,False,False,False,False,...,25-48,No,Yes,No,Yes,No,No,No,Yes,False
2,2,53.85,108.15,2,54.08,0,False,False,False,True,...,0-12,No,Yes,Yes,No,No,No,No,Yes,True
3,45,42.30,1840.75,3,40.91,1,False,False,False,False,...,25-48,No phone service,Yes,No,Yes,Yes,No,No,No,False
4,2,70.70,151.65,0,75.82,0,False,False,False,True,...,0-12,No,No,No,No,No,No,No,Yes,True


## 1. Split first — stratified, before anything touches the target

In [2]:
X = df[ALL_MODEL_COLUMNS].copy()
y = df['churn'].astype(int)

print("Class balance (churn=1):", y.mean().round(4))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train churn rate:", y_train.mean().round(4), " Test churn rate:", y_test.mean().round(4))

Class balance (churn=1): 0.2654
Train: (5634, 24)  Test: (1409, 24)
Train churn rate: 0.2654  Test churn rate: 0.2654


Stratifying on `churn` keeps the ~26.5% churn rate consistent between train and test — important because churn is imbalanced, and an unstratified split could easily give the test set a meaningfully different churn rate purely by chance.

## 2. Build the preprocessing pipeline (scaler + encoder, fit on train only)

We bundle the scaler and encoder into a single fitted `ColumnTransformer` (saved as `models/preprocessor.pkl`) rather than two separate pickle files. This is a deliberate, defensible choice: a single fitted object guarantees the exact same column order and transformation is applied at training time and at live-prediction time in the backend — separate scaler/encoder objects are a common source of subtle train/serve bugs if the column order ever drifts between the two call sites.

In [3]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), NUMERIC_FEATURES),
        ('bin', 'passthrough', BINARY_FEATURES),
        ('nom', OneHotEncoder(handle_unknown='ignore', sparse_output=False), NOMINAL_CATEGORICAL_FEATURES),
    ],
    verbose_feature_names_out=False,
)

preprocessor.fit(X_train)   # <-- fit ONLY on training data

X_train_proc = preprocessor.transform(X_train)
X_test_proc  = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()
X_train_proc = pd.DataFrame(X_train_proc, columns=feature_names, index=X_train.index)
X_test_proc  = pd.DataFrame(X_test_proc,  columns=feature_names, index=X_test.index)

print("Processed feature count:", X_train_proc.shape[1])
X_train_proc.head()

Processed feature count: 48


,tenure,monthly_charges,total_charges,total_services,avg_monthly_spend_ratio,contract_ordinal,senior_citizen,partner,dependents,paperless_billing,...,tech_support_No internet service,tech_support_Yes,streaming_tv_No,streaming_tv_No internet service,streaming_tv_Yes,streaming_movies_No,streaming_movies_No internet service,streaming_movies_Yes,phone_service_No,phone_service_Yes
3738,0.101018,-0.52005,-0.26243,0.509147,-0.537933,-0.829672,False,False,False,False,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0
3151,-0.714014,0.339074,-0.50401,-0.569249,0.393419,-0.829672,False,True,True,False,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
4860,-0.795517,-0.806977,-0.750465,0.509147,-0.644137,1.566828,False,True,True,False,...,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3867,-0.265746,0.286001,-0.17282,1.048345,0.278613,1.566828,False,True,False,True,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
3810,-1.284536,-0.674294,-0.990156,-1.108447,-0.672591,-0.829672,False,True,True,False,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0


## 3. Which models need scaling?

| Needs scaling | Doesn't need scaling |
|---|---|
| Logistic Regression — coefficients are sensitive to feature magnitude; unscaled features distort the regularization penalty | Random Forest — splits on raw thresholds per feature, magnitude-invariant |
| SVM — distance-based margin optimization is magnitude-sensitive | XGBoost / Gradient Boosting — same reason as Random Forest, split-based |
| KNN — literally a distance metric between raw feature vectors | Decision Trees generally |

We scale numeric features for **all** models here for pipeline simplicity (one saved preprocessor for every model) — this is harmless for tree-based models (monotonic transforms don't change their splits) and required for the linear/distance-based models, so a single shared pipeline is the right tradeoff over maintaining two separate preprocessing paths.

## 4. Handle class imbalance

In [4]:
print("Training set class counts:")
print(y_train.value_counts())
print(f"Imbalance ratio: {(y_train==0).sum() / (y_train==1).sum():.2f} : 1 (retained : churned)")

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_proc, y_train)

print("\nAfter SMOTE (training set only — test set is untouched):")
print(y_train_smote.value_counts())

Training set class counts:
churn
0    4139
1    1495
Name: count, dtype: int64
Imbalance ratio: 2.77 : 1 (retained : churned)



After SMOTE (training set only — test set is untouched):
churn
0    4139
1    4139
Name: count, dtype: int64


**Decision:** we apply SMOTE to the *training set only*, after the train/test split and after scaling/encoding — never to the test set, and never before the split. Applying SMOTE before splitting would leak synthetic near-duplicates of test-set-adjacent points into training, inflating test performance artificially. We carry both the original (imbalanced, `class_weight='balanced'`-compatible) training set and the SMOTE-balanced training set into Phase 5, and let the model comparison decide empirically which handles this dataset's imbalance better — rather than assuming.

## 5. Save processed, reproducible datasets

In [5]:
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

X_train_proc.to_csv('../data/processed/X_train.csv', index=False)
X_test_proc.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

pd.DataFrame(X_train_smote, columns=feature_names).to_csv('../data/processed/X_train_smote.csv', index=False)
pd.Series(y_train_smote, name='churn').to_csv('../data/processed/y_train_smote.csv', index=False)

joblib.dump(preprocessor, '../models/preprocessor.pkl')
joblib.dump(list(feature_names), '../models/feature_names.pkl')

print("Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv")
print("Saved: X_train_smote.csv, y_train_smote.csv")
print("Saved: models/preprocessor.pkl, models/feature_names.pkl")

Saved: X_train.csv, X_test.csv, y_train.csv, y_test.csv
Saved: X_train_smote.csv, y_train_smote.csv
Saved: models/preprocessor.pkl, models/feature_names.pkl
